In [9]:
from __future__ import annotations

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import requests
from IPython.display import display
from google.colab import userdata


In [10]:
PARQUET_PATH = Path("/content/sessions_lang_transcript.parquet")

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = "tencent/hy-mt2-30b-a3b"

TARGET_SESSION_ID = 140595413
N_SEGMENTS = 30

REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2
TEMPERATURE = 0
REASONING_ENABLED = True

TRANSIENT_STATUS_CODES = {408, 409, 429, 500, 502, 503, 504}

PARQUET_COLUMNS = [
    "gamesession_id",
    "game_name",
    "model_type",
    "lang_detected",
    "lang_probability",
    "transcript_segments",
]

LANGUAGE_ALIASES = {
    "english": "en",
    "russian": "ru",
    "indonesian": "id",
    "german": "de",
    "french": "fr",
    "spanish": "es",
    "portuguese": "pt",
    "ukrainian": "uk",
    "polish": "pl",
    "japanese": "ja",
    "korean": "ko",
    "chinese": "zh",
    "italian": "it",
    "dutch": "nl",
    "turkish": "tr",
    "arabic": "ar",
    "hindi": "hi",
    "thai": "th",
    "vietnamese": "vi",
    "malay": "ms",
}


In [11]:
def normalize_language(value: Any) -> str:
    language = str(value).strip().lower()
    return LANGUAGE_ALIASES.get(language, language)


def load_sessions(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Parquet file not found: {path}")

    try:
        return pd.read_parquet(path, columns=PARQUET_COLUMNS)
    except Exception as exc:
        raise RuntimeError(f"Failed to read parquet file: {exc}") from exc


def get_session_row(dataframe: pd.DataFrame, session_id: int) -> pd.Series:
    matches = dataframe.loc[dataframe["gamesession_id"] == session_id]

    if matches.empty:
        raise ValueError(f"Session {session_id} not found")

    if len(matches) != 1:
        raise ValueError(
            f"Expected one row for session {session_id}, found {len(matches)}"
        )

    return matches.iloc[0]


df = load_sessions(PARQUET_PATH)
session_row = get_session_row(df, TARGET_SESSION_ID)

segments = session_row["transcript_segments"]

if segments is None:
    raise ValueError("transcript_segments is empty")

internal_lang = normalize_language(session_row["lang_detected"])

print("Session ID          :", session_row["gamesession_id"])
print("Game                :", session_row["game_name"])
print("Internal model      :", session_row["model_type"])
print("Internal language   :", internal_lang)
print("Internal probability:", session_row["lang_probability"])
print("Total segments      :", len(segments))
print("Segments to analyze :", min(N_SEGMENTS, len(segments)))


Session ID          : 140595413
Game                : Counter Strike2
Internal model      : gen10
Internal language   : ru
Internal probability: 0.9873
Total segments      : 714
Segments to analyze : 30


In [12]:
def get_openrouter_api_key() -> str:
    api_key = userdata.get("OPENROUTER_API_KEY")

    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not available in Colab Secrets")

    return api_key


HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update(
    {
        "Authorization": f"Bearer {get_openrouter_api_key()}",
        "Content-Type": "application/json",
    }
)


def build_language_detection_prompt(text: str) -> str:
    return f"""
Analyze every lexical word in the transcript segment below.

For every word, identify its language using an ISO 639-1 code where possible.

Examples:
- en = English
- ru = Russian
- id = Indonesian
- de = German
- fr = French
- es = Spanish
- pt = Portuguese
- uk = Ukrainian
- pl = Polish
- ja = Japanese
- ko = Korean
- zh = Chinese

Rules:
1. Classify every lexical word.
2. Preserve the original word order.
3. Preserve each word as closely as possible to the transcript.
4. Do not add words.
5. Infer language only from the transcript text.
6. Do not use any external language label.
7. Ignore punctuation-only tokens.
8. Return only valid JSON.
9. Do not calculate percentages.

Transcript:
{text}

Return exactly this structure:

{{
  "words": [
    {{
      "word": "example",
      "language": "en"
    }}
  ]
}}
""".strip()


def extract_json_object(content: str) -> dict[str, Any]:
    cleaned = content.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)

        if not match:
            raise ValueError("LLM response does not contain a valid JSON object")

        parsed = json.loads(match.group(0))

    if not isinstance(parsed, dict):
        raise ValueError("LLM response must be a JSON object")

    return parsed


def validate_prediction(prediction: dict[str, Any]) -> list[dict[str, str]]:
    words = prediction.get("words")

    if not isinstance(words, list):
        raise ValueError("LLM response does not contain a valid words list")

    normalized = []

    for index, item in enumerate(words):
        if not isinstance(item, dict):
            raise ValueError(f"words[{index}] is not an object")

        word = str(item.get("word", "")).strip()
        language = normalize_language(item.get("language", ""))

        if not word:
            continue

        if not language:
            raise ValueError(f"words[{index}] has no language")

        normalized.append(
            {
                "word": word,
                "language": language,
            }
        )

    if not normalized:
        raise ValueError("LLM returned no lexical words")

    return normalized


def detect_word_languages(text: str) -> dict[str, Any]:
    text = str(text).strip()

    if not text:
        raise ValueError("Text cannot be empty")

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {
                "role": "user",
                "content": build_language_detection_prompt(text),
            }
        ],
        "temperature": TEMPERATURE,
        "reasoning": {
            "enabled": REASONING_ENABLED,
        },
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = HTTP_SESSION.post(
                OPENROUTER_API_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            if response.status_code in TRANSIENT_STATUS_CODES:
                response.raise_for_status()

            response.raise_for_status()

            response_json = response.json()
            content = response_json["choices"][0]["message"]["content"]

            prediction = extract_json_object(content)
            words = validate_prediction(prediction)

            return {
                "words": words,
                "raw_content": content,
            }

        except (
            requests.RequestException,
            KeyError,
            IndexError,
            TypeError,
            ValueError,
            json.JSONDecodeError,
        ) as exc:
            last_error = exc

            if attempt == MAX_RETRIES:
                break

            time.sleep(RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1)))

    raise RuntimeError(
        f"LLM request failed after {MAX_RETRIES} attempts: {last_error}"
    )


In [13]:
def calculate_language_stats(
    predicted_words: Iterable[dict[str, str]],
) -> tuple[dict[str, int], dict[str, float], dict[str, list[str]]]:
    words = list(predicted_words)

    if not words:
        return {}, {}, {}

    counts = Counter(item["language"] for item in words)
    total_words = len(words)

    percentages = {
        language: count / total_words * 100
        for language, count in counts.items()
    }

    words_by_language = defaultdict(list)

    for item in words:
        words_by_language[item["language"]].append(item["word"])

    return dict(counts), percentages, dict(words_by_language)


def analyze_session_segments(
    segments: Iterable[dict[str, Any]],
    internal_lang: str,
    n_segments: int,
) -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    internal_lang = normalize_language(internal_lang)

    all_results = []
    segment_details = {}
    errors = []

    segments_to_process = list(segments)[:n_segments]

    for segment_index, segment in enumerate(segments_to_process, start=1):
        segment_name = f"segment {segment_index}"
        segment_text = str(segment.get("text", "")).strip()

        if not segment_text:
            errors.append(
                {
                    "segment": segment_name,
                    "text": "",
                    "error": "Empty segment text",
                }
            )
            continue

        print(
            f"[{segment_index}/{len(segments_to_process)}] Processing {segment_name}"
        )

        try:
            prediction = detect_word_languages(segment_text)
            predicted_words = prediction["words"]

            counts, percentages, words_by_language = calculate_language_stats(
                predicted_words
            )

            segment_details[segment_name] = {
                "segment_index": segment_index,
                "text": segment_text,
                "words": predicted_words,
                "counts": counts,
                "percentages": percentages,
                "raw_llm_content": prediction["raw_content"],
            }

            for language, percentage in percentages.items():
                detected_words = words_by_language.get(language, [])
                is_internal_language = language == internal_lang

                all_results.append(
                    {
                        "segment_index": segment_index,
                        "segment": segment_name,
                        "text": segment_text,
                        "language": language,
                        "percentage": percentage,
                        "word_count": counts[language],
                        "detected_words": ", ".join(detected_words),
                        "is_internal_language": is_internal_language,
                        "mismatch_words": (
                            ""
                            if is_internal_language
                            else ", ".join(detected_words)
                        ),
                    }
                )

        except Exception as exc:
            errors.append(
                {
                    "segment": segment_name,
                    "text": segment_text,
                    "error": str(exc),
                }
            )

    result_df = pd.DataFrame(all_results)
    errors_df = pd.DataFrame(errors)

    if result_df.empty:
        return result_df, segment_details, errors_df

    result_df = (
        result_df.sort_values(
            by=[
                "is_internal_language",
                "language",
                "percentage",
                "segment_index",
            ],
            ascending=[
                True,
                True,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return result_df, segment_details, errors_df


In [14]:
result_df, segment_details, errors_df = analyze_session_segments(
    segments=segments,
    internal_lang=internal_lang,
    n_segments=N_SEGMENTS,
)

display_df = result_df.copy()

if not display_df.empty:
    display_df["percentage"] = display_df["percentage"].map(
        lambda value: f"{value:.1f}%"
    )

    display_df = display_df[
        [
            "segment",
            "text",
            "language",
            "percentage",
            "word_count",
            "mismatch_words",
        ]
    ]

print()
print("Session ID       :", session_row["gamesession_id"])
print("Internal language:", internal_lang)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(display_df)

if not errors_df.empty:
    display(errors_df)


[1/30] Processing segment 1
[2/30] Processing segment 2
[3/30] Processing segment 3
[4/30] Processing segment 4
[5/30] Processing segment 5
[6/30] Processing segment 6
[7/30] Processing segment 7
[8/30] Processing segment 8
[9/30] Processing segment 9
[10/30] Processing segment 10
[11/30] Processing segment 11
[12/30] Processing segment 12
[13/30] Processing segment 13
[14/30] Processing segment 14
[15/30] Processing segment 15
[16/30] Processing segment 16
[17/30] Processing segment 17
[18/30] Processing segment 18
[19/30] Processing segment 19
[20/30] Processing segment 20
[21/30] Processing segment 21
[22/30] Processing segment 22
[23/30] Processing segment 23
[24/30] Processing segment 24
[25/30] Processing segment 25
[26/30] Processing segment 26
[27/30] Processing segment 27
[28/30] Processing segment 28
[29/30] Processing segment 29
[30/30] Processing segment 30

Session ID       : 140595413
Internal language: ru


,segment,text,language,percentage,word_count,mismatch_words
0,segment 14,"уходить то какая разница то посиди посидим посидим посидим что мужики как у вас дела чем занимались это у меня вон хрюндель уже есть собирать жени го когда я приличу будем играть вместе вообще каждый день готов играть в эту прекрасную игру йоу на мемфисе плюс мы спокойно вступили и можем все вступить в семью к диману и с ними играть я считаю плюс даже если меня что-то не сложится, вы можете с",en,42.5%,31,"йоу, на, мемфисе, плюс, мы, спокойно, вступили, и, можем, все, вступить, в, семью, к, диману, и, с, ними, играть, я, считаю, плюс, даже, если, меня, что-то, не, сложится, вы, можете, с"
1,segment 9,"до свидания, я хочу поработать у вас 600 долларов на шахте!",en,18.2%,2,"600, долларов"
2,segment 10,"Здорово мужики, как у вас дела? меня GTA уже несколько лета, ну так заходи, немедленно на memphis, по правому коду плюс w у тебя будет плюс 5 тысячи долларов, плюс у тебя еще будет 7 невипки, не забывай.",en,15.8%,6,"GTA, memphis, w, 5, 7, невипки"
3,segment 11,"Так, ладно, зарабатывайте 600 долларов на шахте, а я же вроде бы заработал, нет?",en,14.3%,2,"600, долларов"
4,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.",en,13.3%,2,"50, 50"
5,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? Нет, на дорожке?",en,13.3%,2,"50, 50"
6,segment 2,"Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.",en,10.0%,1,Дэнс
7,segment 27,мог сделать ну мы никогда не пойдем и не идем точно я вообще с если чё и бэкнусь блин ну грустно на самом деле что решили больше никакой пока проект вы не идти но если чё приходите на маджестик на этот на мемфис вайнот как говорится ладно я папки поэтому отошел давай все наилучшего а я чё знаю что ты на матч перешел ну йоу йоу женёк на церковь меня вид,en,8.5%,6,"маджестик, мемфис, вайнот, матч, йоу, йоу"
8,segment 8,"Нет, как будто бы начальник спал. место выхода да мне нужно вместо выхода подожди что ты пишешь хрюндель ты долбоеб хрюндель вот скажи мне честно ты дурачок или что вот если тебе ну не купили гта в детстве ты не попробовал гта это не значит что гта его ну не очень согласись не может быть что гта для тебя полная фигня я вообще никогда не поверю я знаю только одно что тебе нравится гта в целом так Дополнительное",en,6.4%,5,"GTA, GTA, GTA, GTA, GTA"
9,segment 13,Majestic вперед! ну мужики алгава джестик получается в штрихе плюсядера у мучачи с как дела жду не дождольсь когда пройду 7 дней буду обменивать скины на бабочку ручная роспись вы говнюки вы все сейчас будете с бабочками а мне когда подарите ну придать прит зузи лики ну не уходи у меня у меня потери пока тут но нет что,en,5.3%,3,"Majestic, 7, скины"


In [15]:
mismatch_df = result_df.loc[
    ~result_df["is_internal_language"]
].copy()

if not mismatch_df.empty:
    mismatch_df["percentage"] = mismatch_df["percentage"].map(
        lambda value: f"{value:.1f}%"
    )

    mismatch_df = mismatch_df[
        [
            "segment",
            "text",
            "language",
            "percentage",
            "word_count",
            "mismatch_words",
        ]
    ]

print("Internal language:", internal_lang)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(mismatch_df)


Internal language: ru


,segment,text,language,percentage,word_count,mismatch_words
0,segment 14,"уходить то какая разница то посиди посидим посидим посидим что мужики как у вас дела чем занимались это у меня вон хрюндель уже есть собирать жени го когда я приличу будем играть вместе вообще каждый день готов играть в эту прекрасную игру йоу на мемфисе плюс мы спокойно вступили и можем все вступить в семью к диману и с ними играть я считаю плюс даже если меня что-то не сложится, вы можете с",en,42.5%,31,"йоу, на, мемфисе, плюс, мы, спокойно, вступили, и, можем, все, вступить, в, семью, к, диману, и, с, ними, играть, я, считаю, плюс, даже, если, меня, что-то, не, сложится, вы, можете, с"
1,segment 9,"до свидания, я хочу поработать у вас 600 долларов на шахте!",en,18.2%,2,"600, долларов"
2,segment 10,"Здорово мужики, как у вас дела? меня GTA уже несколько лета, ну так заходи, немедленно на memphis, по правому коду плюс w у тебя будет плюс 5 тысячи долларов, плюс у тебя еще будет 7 невипки, не забывай.",en,15.8%,6,"GTA, memphis, w, 5, 7, невипки"
3,segment 11,"Так, ладно, зарабатывайте 600 долларов на шахте, а я же вроде бы заработал, нет?",en,14.3%,2,"600, долларов"
4,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.",en,13.3%,2,"50, 50"
5,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? Нет, на дорожке?",en,13.3%,2,"50, 50"
6,segment 2,"Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.",en,10.0%,1,Дэнс
7,segment 27,мог сделать ну мы никогда не пойдем и не идем точно я вообще с если чё и бэкнусь блин ну грустно на самом деле что решили больше никакой пока проект вы не идти но если чё приходите на маджестик на этот на мемфис вайнот как говорится ладно я папки поэтому отошел давай все наилучшего а я чё знаю что ты на матч перешел ну йоу йоу женёк на церковь меня вид,en,8.5%,6,"маджестик, мемфис, вайнот, матч, йоу, йоу"
8,segment 8,"Нет, как будто бы начальник спал. место выхода да мне нужно вместо выхода подожди что ты пишешь хрюндель ты долбоеб хрюндель вот скажи мне честно ты дурачок или что вот если тебе ну не купили гта в детстве ты не попробовал гта это не значит что гта его ну не очень согласись не может быть что гта для тебя полная фигня я вообще никогда не поверю я знаю только одно что тебе нравится гта в целом так Дополнительное",en,6.4%,5,"GTA, GTA, GTA, GTA, GTA"
9,segment 13,Majestic вперед! ну мужики алгава джестик получается в штрихе плюсядера у мучачи с как дела жду не дождольсь когда пройду 7 дней буду обменивать скины на бабочку ручная роспись вы говнюки вы все сейчас будете с бабочками а мне когда подарите ну придать прит зузи лики ну не уходи у меня у меня потери пока тут но нет что,en,5.3%,3,"Majestic, 7, скины"


In [16]:
for segment_name, detail in segment_details.items():
    print("=" * 100)
    print(segment_name)
    print(detail["text"])
    print()

    words_df = pd.DataFrame(detail["words"])

    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
        "display.max_rows", None,
    ):
        display(words_df)

    print("Counts     :", detail["counts"])
    print("Percentages:", detail["percentages"])
    print()


segment 1
кушать готовил очень вкусненько но приходите присаживайся реально надо подождем блин гта да ну на худа хики еще азиатик здорово привет моя киса я если что натурал сразу говорю я с азиатиком не заигрываю никогда я натурал уверенный в себе мужчина йоу я натурал уверенный



,word,language
0,кушать,ru
1,готовил,ru
2,очень,ru
3,вкусненько,ru
4,но,ru
5,приходите,ru
6,присаживайся,ru
7,реально,ru
8,надо,ru
9,подождем,ru


Counts     : {'ru': 42, 'en': 2, 'ja': 1}
Percentages: {'ru': 93.33333333333333, 'en': 4.444444444444445, 'ja': 2.2222222222222223}

segment 2
Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.



,word,language
0,Вот,ru
1,он,ru
2,какой,ru
3,маленький,ru
4,сидит,ru
5,сладенький,ru
6,вкусненький,ru
7,и,ru
8,все,ru
9,Дэнс,en


Counts     : {'ru': 9, 'en': 1}
Percentages: {'ru': 90.0, 'en': 10.0}

segment 3
Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.



...

Counts     : {'ru': 65}
Percentages: {'ru': 100.0}

segment 29
которая будет мне мурчать постоянно, ну? Подожди, а че я там че-то не взял?



,word,language
0,которая,ru
1,будет,ru
2,мне,ru
3,мурчать,ru
4,постоянно,ru
5,ну,ru
6,Подожди,ru
7,а,ru
8,че,ru
9,я,ru


Counts     : {'ru': 14}
Percentages: {'ru': 100.0}

segment 30
Путь бомжика в ГТА, что Ну не то чтобы путь бомжика, мужики.



,word,language
0,Путь,ru
1,бомжика,ru
2,в,ru
3,ГТА,ru
4,что,ru
5,Ну,ru
6,не,ru
7,то,ru
8,быть,ru
9,путь,ru


Counts     : {'ru': 12}
Percentages: {'ru': 100.0}

